# 04 Analysis
Post-simulation analysis: outage filtering, per-city/ISP/SRG failure summaries.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import pickle
import yaml
from src.network.build import prepare_edge_df
from src.analysis.analysis import (
    filter_outages,
    create_city_fail_df,
    count_fails,
    fail_summary,
    record_srg_failures,
    create_srg_isp_df,
    city_outage_dataframe,
)

In [ ]:
# Reload graph and node data for analysis
with open("../config.yaml", "r") as file:
    config = yaml.safe_load(file)

# import directories and load
NODE_DIR = config["paths"]["node_list"]
EDGE_DIR = config["paths"]["edge_list"]
IXP_DIR = config["paths"]["ixp_list"]
SRG_DIR = config["paths"]["srg_list"]
SITE_DIR = config["paths"]["site_list"]
GRAPH_DIR = config["paths"]["graph_folder"]
TABULAR_DIR = config["paths"]["tabular_folder"]
RESULTS_DIR = config["paths"]["results_folder"]

node_df = pd.read_csv(NODE_DIR)
edge_df = pd.read_csv(EDGE_DIR)
edge_df = prepare_edge_df(edge_df)
srg_city_df = pd.read_csv(SRG_DIR)
ixp_df = pd.read_csv(IXP_DIR)
site_df = pd.read_csv(SITE_DIR, comment='#')
site_list = list(site_df['site_id'])

# load graph object
with open(f"{GRAPH_DIR}/G_full.pickle", "rb") as f:
    G_full = pickle.load(f)

# load srlg df
srg_df = pd.read_csv(f'{TABULAR_DIR}/srg_c.csv')

In [18]:
# Load simulation results
single_df = pd.read_csv(f'{TABULAR_DIR}/single.csv')
double_df = pd.read_csv(f'{TABULAR_DIR}/double.csv')

print('Loaded simulation results.')
print('Double failure pairs:', len(double_df) -1, 'scenarios')

Loaded simulation results.
Double failure pairs: 1275 scenarios


In [19]:
# Create df filtered to only include outages
outages_df = filter_outages(double_df)
print('Disconnect occurrences:', len(outages_df))

# global only
global_count = (outages_df['n_components'] > 1).sum()
print(f'Global disconnects: {global_count}')
outages_df.head()

Disconnect occurrences: 111
Global disconnects: 15


,nodes_total,edges_total,n_components,largest_cc_frac,in_lcc,present_sites,diameter,avg_shortest_path,avg_cluster_coeff,aarnet_nodes,...,vocus_present_sites,vocus_diameter,vocus_avg_shortest_path,vocus_avg_cluster_coeff,scenario,srg_id_1,srg_id_2,edges_removed,edges_removed_1,edges_removed_2
0,67,160,4,0.895522,12,"['ADL_1', 'BNE_1', 'CBR_1', 'DRW_1', 'HBA_1', ...",inf,inf,0.250746,11,...,"['ADL_1', 'BNE_1', 'CBR_1', 'DRW_1', 'MEL_1', ...",inf,inf,0.530303,double removal,HBA_LST_TER_1,HBA_LST_TER_2,11.0,5.0,6.0
1,67,160,3,0.910448,13,"['ADL_1', 'BNE_1', 'CBR_1', 'DRW_1', 'HBA_1', ...",inf,inf,0.250746,11,...,"['ADL_1', 'BNE_1', 'CBR_1', 'DRW_1', 'MEL_1', ...",inf,inf,0.530303,double removal,HBA_LST_TER_1,MEL_LST_SUB_2,11.0,5.0,6.0
2,67,161,2,0.925373,13,"['ADL_1', 'BNE_1', 'CBR_1', 'DRW_1', 'HBA_1', ...",inf,inf,0.250746,11,...,"['ADL_1', 'BNE_1', 'CBR_1', 'DRW_1', 'MEL_1', ...",inf,inf,0.530303,double removal,HBA_LST_TER_2,MEL_LST_SUB_2,12.0,6.0,6.0
3,67,161,3,0.940299,12,"['ADL_1', 'BNE_1', 'CBR_1', 'DRW_1', 'HBA_1', ...",inf,inf,0.252736,11,...,"['ADL_1', 'BNE_1', 'CBR_1', 'DRW_1', 'MEL_1', ...",inf,inf,0.530303,double removal,MEL_LST_SUB_1,MEL_LST_SUB_2,10.0,4.0,6.0
4,67,153,4,0.955224,13,"['ADL_1', 'BNE_1', 'CBR_1', 'DRW_1', 'HBA_1', ...",inf,inf,0.200498,11,...,"['ADL_1', 'BNE_1', 'DRW_1', 'HBA_1', 'MEL_1', ...",inf,inf,0.393939,double removal,MEL_CBR_TER_2,SYD_CBR_TER_2,18.0,9.0,9.0


In [20]:
# Create a per-failure DataFrame (failure, ISP, city)
# Specifies each failure with an ID, and lists the cities disconnected and for each ISP
fail_df = create_city_fail_df(outages_df, node_df)

print('Failure rows:', len(fail_df))
fail_df.head()

Failure rows: 140


,failure_id,isp,city,global,srg_1,srg_2
0,F0,aarnet,HBA_1,True,HBA_LST_TER_1,HBA_LST_TER_2
1,F0,abb,HBA_1,True,HBA_LST_TER_1,HBA_LST_TER_2
2,F0,optus,HBA_1,True,HBA_LST_TER_1,HBA_LST_TER_2
3,F0,superloop,HBA_1,True,HBA_LST_TER_1,HBA_LST_TER_2
4,F0,telstra,HBA_1,True,HBA_LST_TER_1,HBA_LST_TER_2


In [21]:
# Get per failure stats (count how many cities, and ISPs affected by these failures)
city_isp_fail_counts = count_fails(fail_df)
city_isp_fail_counts.head()

,failure_id,n_cities,n_isps,is_global
0,F0,1,6,True
1,F1,1,5,True
2,F10,1,1,True
3,F100,1,1,False
4,F101,1,1,False


In [22]:
# Failure summary (key stats for both single and double failures)
single_fail_sum = fail_summary(single_df)
double_fail_sum = fail_summary(double_df)

print(single_fail_sum)
print(double_fail_sum)

{'N_trials': 51, 'N_disconnects': 2, 'N_no_disconnects': 49, 'N_local_disconnects': 2, 'N_global_disconnects': 0}
{'N_trials': 1275, 'N_disconnects': 111, 'N_no_disconnects': 1164, 'N_local_disconnects': 96, 'N_global_disconnects': 15}


In [23]:
# SRLG-level failure counts (number of failures each SRLG were involved in)
srg_sum_df = record_srg_failures(outages_df)

print('SRLGs involved in global outages:', (srg_sum_df['global_outages'] > 0).sum())
srg_sum_df.sort_values('outages', ascending=False).head(10)

SRLGs involved in global outages: 14


,srg_id,outages,global_outages,local_only_outages
1,HBA_LST_TER_2,50,3,47
22,MEL_LST_SUB_2,50,3,47
5,ISA_HGD_TER_1,5,3,2
8,BNE_ROK_TER_1,5,3,2
4,ADL_DRW_TER_1,5,2,3
24,TSV_ROK_TER_1,5,3,2
0,HBA_LST_TER_1,4,3,1
7,DRW_ISA_TER_1,4,1,3
6,TSV_HGD_TER_1,4,2,2
2,MEL_LST_SUB_1,3,3,0


In [24]:
# SRG x ISP membership matrix (matrix showing for each SRLG which ISPs use the SRLG)
srg_vs_isp_df = create_srg_isp_df(single_df)

print(srg_vs_isp_df.shape)
srg_vs_isp_df.head()

(51, 7)


,srg_id,aarnet,abb,optus,superloop,telstra,vocus
0,ADL_DRW_TER_1,1.0,1.0,1.0,1.0,1.0,1.0
1,ADL_SMAP_SUB_1,0.0,1.0,0.0,0.0,1.0,0.0
2,BNE_ROK_TER_1,1.0,0.0,1.0,0.0,1.0,1.0
3,BNE_ROK_TER_2,1.0,0.0,0.0,0.0,0.0,0.0
4,BNE_ROK_TER_3,1.0,0.0,1.0,1.0,0.0,1.0


In [25]:
# City-level outage summary
city_sum_df = city_outage_dataframe(G_full, fail_df, site_list)

city_sum_df.sort_values('site', ascending=True).head(10)

,site,layers,supra_degree_sum,aarnet_degree,abb_degree,optus_degree,superloop_degree,telstra_degree,vocus_degree,ixps,isp_fails,local_fails,global_fails
0,ADL_1,6,37,5,6,5,5,5,5,6,0,0,0
1,BNE_1,6,37,6,3,6,5,5,6,6,0,0,0
2,CBR_1,6,20,3,3,2,2,4,4,2,3,0,1
3,DRW_1,6,20,2,2,2,3,3,4,4,4,1,2
4,HBA_1,6,15,2,2,2,2,2,1,4,123,97,6
13,HGD_1,0,0,0,0,0,0,0,0,0,0,0,0
12,ISA_1,2,6,3,0,0,0,0,3,0,1,0,1
14,LRE_1,0,0,0,0,0,0,0,0,0,0,0,0
8,LST_1,2,6,0,2,4,0,0,0,0,4,0,3
5,MEL_1,6,49,6,8,6,8,7,8,6,0,0,0


In [26]:
# Save analysis outputs
import os
os.makedirs(RESULTS_DIR, exist_ok=True)

fail_df.to_csv(f'{RESULTS_DIR}/failures.csv', index=False)
city_isp_fail_counts.to_csv(f'{RESULTS_DIR}/city_isp_fail_counts.csv', index=False)
srg_sum_df.to_csv(f'{RESULTS_DIR}/srg_summary.csv', index=False)
srg_vs_isp_df.to_csv(f'{RESULTS_DIR}/srg_isp_usage.csv', index=False)
city_sum_df.to_csv(f'{RESULTS_DIR}/city_summary.csv', index=False)
print('Saved.')

Saved.
